In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import os
import time
import random
import requests
import pandas as pd

url = "https://api.gdeltproject.org/api/v2/doc/doc"

ticker = "CVX"
company = '("Chevron" OR "Chevron Corporation")'

start_year = 2020
end_year = 2025

output_file = f"/kaggle/working/gdelt_{ticker.lower()}_2020_2025.csv"

expected_days = {
    2020: 366,
    2021: 365,
    2022: 365,
    2023: 365,
    2024: 366,
    2025: 365
}

In [3]:
if os.path.exists(output_file):

    existing_df = pd.read_csv(output_file)
    existing_df["date_only"] = pd.to_datetime(existing_df["date_only"])

    existing_df = (existing_df.drop_duplicates(subset=["ticker", "date_only"]).sort_values("date_only").reset_index(drop=True))

    print("Existing file found.")
    print("Existing rows:", len(existing_df))

else:
    existing_df = pd.DataFrame(columns=["date_only", "tone", "ticker"])
    print("No existing output file found.")

No existing output file found.


In [4]:
def fetch_gdelt_year(
    year,
    ticker,
    company,
    max_attempts=12,
    base_wait=30
):

    params = {
        "query": company,
        "mode": "timelinetone",
        "format": "json",
        "startdatetime": f"{year}0101000000",
        "enddatetime": f"{year}1231235959"
    }

    for attempt in range(1, max_attempts + 1):
        try:
            response = requests.get(
                url,
                params=params,
                timeout=120,
                headers={
                    "User-Agent":
                    "Mozilla/5.0 academic-research-project/1.0"})

            print(
                f"{year} | attempt {attempt}/{max_attempts} "
                f"| status {response.status_code}")

            if response.status_code == 200:
                data = response.json()
                rows = []

                for series in data.get("timeline", []):
                    for point in series.get("data", []):
                        rows.append({
                            "date_only": point["date"][:8],
                            "tone": point["value"],
                            "ticker": ticker})

                if rows:
                    year_df = pd.DataFrame(rows)
                    year_df["date_only"] = pd.to_datetime(
                        year_df["date_only"],
                        format="%Y%m%d")

                    year_df = (year_df.drop_duplicates(subset=["ticker", "date_only"]).sort_values("date_only").reset_index(drop=True))

                    print(f"{year}: successfully collected " f"{len(year_df)} rows")
                    return year_df
                print(f"{year}: status 200 but no timeline data returned")

            elif response.status_code == 429:
                wait_time = (base_wait * attempt + random.randint(10, 30))
                print(f"Rate limited. Waiting "f"{wait_time} seconds...")
                time.sleep(wait_time)

            else:
                wait_time = (base_wait + random.randint(10, 30))
                print(f"Request failed. Waiting "f"{wait_time} seconds...")
                time.sleep(wait_time)

        except requests.exceptions.RequestException as error:
            wait_time = (base_wait * attempt + random.randint(10, 30))

            print("Request error:", error)
            print(f"Waiting {wait_time} seconds...")
            time.sleep(wait_time)

        except ValueError as error:
            wait_time = (base_wait + random.randint(10, 30))
            print("JSON parsing error:", error)
            print(f"Waiting {wait_time} seconds...")
            time.sleep(wait_time)

    print(f"{year}: failed after "f"{max_attempts} attempts")
    return pd.DataFrame()

In [5]:
minimum_days_per_year = 340
year_counts = {}
if not existing_df.empty:
    year_counts = (existing_df.groupby(existing_df["date_only"].dt.year).size().to_dict())

years_to_fetch = []
for year in range(start_year, end_year + 1):
    collected_days = year_counts.get(year, 0)
    if collected_days < minimum_days_per_year:
        years_to_fetch.append(year)
        print(f"{year}: incomplete "f"({collected_days} rows) → will retrieve")

    else:
        print(f"{year}: already available "f"({collected_days} rows) → skipped")
print("\nYears to retrieve:", years_to_fetch)

2020: incomplete (0 rows) → will retrieve
2021: incomplete (0 rows) → will retrieve
2022: incomplete (0 rows) → will retrieve
2023: incomplete (0 rows) → will retrieve
2024: incomplete (0 rows) → will retrieve
2025: incomplete (0 rows) → will retrieve

Years to retrieve: [2020, 2021, 2022, 2023, 2024, 2025]


In [6]:
combined_df = existing_df.copy()
for year in years_to_fetch:

    print(f"\n{'=' * 50}")
    print(f"Retrieving {ticker} for {year}")
    print(f"{'=' * 50}")

    year_df = fetch_gdelt_year(
        year=year,
        ticker=ticker,
        company=company,
        max_attempts=12,
        base_wait=30)

    if not year_df.empty:
        combined_df = pd.concat(
            [combined_df, year_df],
            ignore_index=True)

        combined_df["date_only"] = pd.to_datetime(
            combined_df["date_only"])

        combined_df = (
            combined_df
            .drop_duplicates(
                subset=["ticker", "date_only"],
                keep="last")
            .sort_values(["ticker", "date_only"])
            .reset_index(drop=True))

        combined_df.to_csv(
            output_file,
            index=False)

        print(
            f"Checkpoint saved after {year}: "
            f"{output_file}")

    else:
        print(
            f"{year} was not collected. "
            f"Continuing to the next year.")
    # Rest before the next yearly request
    time.sleep(120)


Retrieving CVX for 2020
2020 | attempt 1/12 | status 429
Rate limited. Waiting 46 seconds...
2020 | attempt 2/12 | status 429
Rate limited. Waiting 83 seconds...
2020 | attempt 3/12 | status 429
Rate limited. Waiting 115 seconds...
2020 | attempt 4/12 | status 429
Rate limited. Waiting 144 seconds...
2020 | attempt 5/12 | status 200
2020: successfully collected 365 rows
Checkpoint saved after 2020: /kaggle/working/gdelt_cvx_2020_2025.csv


/tmp/ipykernel_58/251298458.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat(



Retrieving CVX for 2021
2021 | attempt 1/12 | status 429
Rate limited. Waiting 41 seconds...
2021 | attempt 2/12 | status 429
Rate limited. Waiting 76 seconds...
2021 | attempt 3/12 | status 429
Rate limited. Waiting 115 seconds...
2021 | attempt 4/12 | status 200
2021: successfully collected 365 rows
Checkpoint saved after 2021: /kaggle/working/gdelt_cvx_2020_2025.csv

Retrieving CVX for 2022
2022 | attempt 1/12 | status 429
Rate limited. Waiting 54 seconds...
2022 | attempt 2/12 | status 429
Rate limited. Waiting 72 seconds...
2022 | attempt 3/12 | status 429
Rate limited. Waiting 105 seconds...
2022 | attempt 4/12 | status 200
2022: successfully collected 365 rows
Checkpoint saved after 2022: /kaggle/working/gdelt_cvx_2020_2025.csv

Retrieving CVX for 2023
2023 | attempt 1/12 | status 429
Rate limited. Waiting 45 seconds...
2023 | attempt 2/12 | status 200
2023: successfully collected 364 rows
Checkpoint saved after 2023: /kaggle/working/gdelt_cvx_2020_2025.csv

Retrieving CVX for 

In [8]:
if os.path.exists(output_file):

    final_df = pd.read_csv(output_file)
    final_df["date_only"] = pd.to_datetime(
        final_df["date_only"])

    coverage = (
        final_df
        .groupby(final_df["date_only"].dt.year)
        .size()
        .rename("rows")
        .reset_index()
        .rename(columns={"date_only": "year"}))
    display(coverage)

    print("Total rows:", len(final_df))
    print("First date:", final_df["date_only"].min())
    print("Last date:", final_df["date_only"].max())

    print(
        "Duplicate ticker-dates:",
        final_df.duplicated(
            subset=["ticker", "date_only"]
        ).sum())

else:
    print("No output file was produced.")

,year,rows
0,2020,365
1,2021,365
2,2022,365
3,2023,364
4,2024,366
5,2025,348


Total rows: 2173
First date: 2020-01-01 00:00:00
Last date: 2025-12-31 00:00:00
Duplicate ticker-dates: 0
